In [2]:
#import packages

import numpy as np
import matplotlib.pyplot as plt
from numpy import linalg as LA
import time

import scipy.optimize as opt
from scipy.optimize import minimize
from numpy.random import rand


In [3]:
#auxiliary functions

#Kronecker delta
def kron_del(x,y):
    if x==y:
        return 1
    return 0

#Check if the solution is physical, i.e. if the damping is positive in all the domain
def check_pos(x):
    n=1000
    b=x.reshape(2*N+1,2*N+1)

    x_segment=np.linspace(0,np.pi,n)
    y_segment=np.linspace(0,np.pi,n)

    y_segment=np.array([y_segment]*n).reshape(n**2)
    x_segment=np.array([x for item in x_segment for x in repeat(item, n)])

    damp_spat=np.zeros(n**2)

    for i in range (2*N+1):
        for j in range (2*N+1):
            damp_spat+=b[i][j]*np.cos(i*x_segment)*np.cos(j*y_segment)*(1-kron_del(i,0)/2)*(1-kron_del(j,0)/2)

    return damp_spat

cons = {'type':'ineq', 'fun': check_pos}#dictionary to be used in the minimization function

#Define the matrix for the Jacobian
def matrix(h):
    M=np.zeros((N,N,N,N))
    for l in range (N):
        for k in range (N):
            for n in range (l,N):
                for m in range (k,N):
                    int_test=((n**2+m**2)*(h[abs(n-l),abs(m-k)]+h[n+l,m+k]+h[abs(n-l),m+k]+h[n+l,abs(m-k)])\
                    -n*((n+l)*h[n+l,abs(m-k)]+(n-l)*h[abs(n-l),m+k]+(n+l)*h[n+l,m+k]+(n-l)*h[abs(n-l),abs(m-k)])\
                    -m*((m-k)*h[n+l,abs(m-k)]+(m+k)*h[abs(n-l),m+k]+(m-k)*h[abs(n-l),abs(m-k)]+(m+k)*h[n+l,m+k]))
                    M[l][k][n][m]=int_test
                    M[n][m][l][k]=int_test
    return M

#Define the Jacobian here
def Jac(M):
    mat_aux=np.array([[l**2+k**2 for l in range (N)]for k in range (N)])
    vec_aux=mat_aux.reshape(N**2)
    M_aux=np.diag(vec_aux)
    return np.block([[np.zeros((N**2,N**2)),np.identity(N**2)],[-M_aux,-M.reshape((N**2,N**2))]])

# Minimization function
def objective(x):
    M=matrix(x.reshape(2*N+1,2*N+1))
    J=Jac(M)
    EV=np.real(np.sort(LA.eigvals(J)))
    return EV[len(EV)-3]#exclude the identically zero eigenvalues

In [5]:
#Calculate optimal heterogeneous one

N_min=2
N_max=5

for N in range (N_min,N_max):
    start=time.time()

    n_p0=10**(2)
    eps=0.003
    n_iter=10**(6)

    bounds_min=np.zeros(((2*N+1)**2,2))
    for i in range ((2*N+1)**2):
        bounds_min[i]=[-eps,eps]
    bounds_min[0]=[0,10]

    min_fun=0
    min_x=np.full((2*N+1)**2,-1)
    for i in range (n_p0):
        p0 = -eps + rand((2*N+1)**2)*2*eps
        p0[0]=1
        result_min = minimize(objective, x0=p0, method='SLSQP',bounds=bounds_min, options={'maxiter':n_iter}, tol=10**(-6))
        if result_min.fun<min_fun:
            min_fun=result_min.fun
            min_x=result_min.x

        #print results of minimization
    print(N)
    print(min_fun)
    print(check_pos(min_x))
    print(np.array(repr(min_x)))
            
    end=time.time()
    print(end-start)

2
-0.9999893948673012
[0.24302145 0.24302179 0.24302281 ... 0.24210172 0.24210022 0.24209971]
array([ 1.00252836e+00,  2.97850295e-03,  2.53297354e-03, -5.68717623e-04,
       -1.64140624e-03,  1.04937913e-03, -1.46702692e-03,  2.13883122e-03,
       -2.60534837e-03,  1.92601017e-03,  2.57335089e-03, -2.50543003e-04,
        2.30373754e-03, -4.29531753e-04, -2.26477059e-03, -2.62452645e-03,
       -2.42736421e-03, -1.68584980e-03, -1.22764629e-03, -3.77889602e-04,
        1.65306478e-03, -7.97695186e-04, -2.95812691e-03, -4.79781692e-04,
        1.60494979e-05])
34.2121205329895
3
-0.7593645292567309
[0.17896979 0.17897062 0.1789731  ... 0.18379859 0.1837976  0.18379727]
array([ 7.56363789e-01,  1.45174859e-03, -2.88718683e-03,  1.56439197e-03,
        3.00000000e-03,  2.71855321e-03,  2.82893300e-04, -2.45163681e-03,
       -2.72066621e-03, -2.46211224e-03,  2.99721662e-03, -1.88811620e-03,
        1.12722486e-03, -2.10872303e-03, -2.87307821e-03,  1.37243812e-03,
       -3.00000000e-